# analysis_01 — proposition : un discount factor propre au terme empathique ($\gamma_{\text{autres}} < \gamma_{\text{soi}}$)

**Statut : implémenté le 02/10/2026** (`DQNAgent(gamma_others=...)`, expérience `exp6_social_discount` dans `main_gt_ais.py`, combinaison brute). Le texte ci-dessous est la proposition qui a mené à ce choix. Ce notebook ne lance
aucune simulation et ne lit aucun CSV. Les cellules de code sont des illustrations sur des
nombres ou des tenseurs factices.

**L'intuition.** Aujourd'hui, un seul $\gamma$ actualise la reward combinée
$r_i = (1-\alpha_i)\,p_i + \alpha_i\,\tilde e_i$. L'agent se soucie donc **au même rythme** de son propre
avenir et de l'avenir émotionnel des autres. Or, chez l'humain, on se désintéresse plus vite des
intérêts futurs des autres que des siens. On voudrait donc

$$
\gamma_{\text{autres}} \;\le\; \gamma_{\text{soi}} .
$$

**Plan.**
1. Ce que l'objectif devient (§1)
2. Conséquence : une empathie « au présent » et une incohérence temporelle (§2)
3. Implémentation : un DQN à deux têtes (§3)
4. Brute ou normalisée : ce que fixe chaque version, et le contrôle par α apparié (§4)
5. Pourquoi c'est un test direct du mécanisme dans notre environnement (§5)
6. Conditions possibles, points ouverts, littérature (§6–§7)
7. Quelle valeur pour $\gamma_o$ ? Ce que disent les données humaines (§8)

## 1. L'objectif avec deux discount factors

L'agent $i$ maximise aujourd'hui

$$
G_i = \sum_{k\ge 0} \gamma^k \big[(1-\alpha_i)\,p_i(t+k) + \alpha_i\,\tilde e_i(t+k)\big].
$$

Avec deux discount factors :

$$
G_i = (1-\alpha_i)\sum_{k\ge 0} \gamma_s^{\,k}\, p_i(t+k) \;+\; \alpha_i \sum_{k\ge 0} \gamma_o^{\,k}\, \tilde e_i(t+k),
\qquad \gamma_o \le \gamma_s ,
$$

où $\gamma_s$ = « soi » et $\gamma_o$ = « autres ». La reward étant **linéaire** en $p$ et $\tilde e$, la valeur
d'action se décompose exactement :

$$
Q_i(s,a) = (1-\alpha_i)\,Q^{p}_i(s,a) + \alpha_i\,Q^{e}_i(s,a),
$$

où $Q^p$ est la valeur du flux personnel actualisée à $\gamma_s$ et $Q^e$ celle du flux empathique
actualisée à $\gamma_o$. **Si $\gamma_o = \gamma_s$, c'est exactement le modèle actuel** : la proposition englobe donc
toutes les expériences existantes comme cas particulier.

## 2. Conséquence : le poids de l'empathie décroît avec l'horizon

Vu depuis l'instant $t$, le poids relatif de l'empathie pour un événement à $k$ pas dans le futur vaut

$$
\alpha_{\text{eff}}(k) = \frac{\alpha\,\gamma_o^{\,k}}{(1-\alpha)\,\gamma_s^{\,k} + \alpha\,\gamma_o^{\,k}} .
$$

Avec $\gamma_o < \gamma_s$, $\alpha_{\text{eff}}(k)$ part de $\alpha$ à $k=0$ et tend vers 0. **L'agent est empathique pour le
présent, égoïste pour le lointain.** C'est exactement l'intuition de départ.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SURFACE = "#fcfcfb"
INK = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#898781", "grid": "#e1e0d9"}
ORDINAL_BLUE = ["#c3dcf8", "#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "figure.dpi": 110,
    "axes.edgecolor": INK["grid"], "axes.labelcolor": INK["secondary"],
    "xtick.color": INK["muted"], "ytick.color": INK["muted"],
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": INK["grid"], "grid.linewidth": 0.6,
})

ALPHA = 0.5
GAMMA_SELF = 0.99
GAMMA_OTHERS = [0.99, 0.98, 0.95, 0.9, 0.8]   # 0.99 = modèle actuel

def alpha_eff(k, alpha, g_s, g_o):
    # poids relatif du terme empathique pour un événement à k pas
    w_o = alpha * g_o ** k
    w_s = (1 - alpha) * g_s ** k
    return w_o / (w_s + w_o)

k = np.arange(0, 201)
fig, ax = plt.subplots(figsize=(7, 3.6))
colors = ["#898781"] + ORDINAL_BLUE[2:]
for g_o, c in zip(GAMMA_OTHERS, colors):
    label = rf"$\gamma_o$ = {g_o}" + ("  (actuel)" if g_o == GAMMA_SELF else "")
    ax.plot(k, alpha_eff(k, ALPHA, GAMMA_SELF, g_o), color=c, lw=2, label=label)
ax.axvline(10, color=INK["muted"], lw=0.8, ls=":")
ax.text(11, 0.47, "fenêtre mémoire\ndes émotions (10 pas)", color=INK["muted"], fontsize=8, va="top")
ax.set_xlabel("horizon k (pas)")
ax.set_ylabel(r"$\alpha_{\mathrm{eff}}(k)$")
ax.set_title(rf"Poids effectif de l'empathie selon l'horizon ($\alpha$={ALPHA}, $\gamma_s$={GAMMA_SELF})",
             loc="left", fontsize=10, color=INK["primary"])
ax.set_ylim(0, 0.55)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

**Incohérence temporelle (à assumer ou à discuter).** Avec deux $\gamma$ différents, le poids
relatif entre « moi » et « les autres » à une date donnée **change selon le moment d'où l'on
regarde**. Au pas $t$, l'agent planifie d'être généreux au pas $t+50$ ($\alpha_{\text{eff}}$ faible vu de loin). Arrivé à $t+50$,
cet événement est devenu présent et pèse $\alpha$ : l'agent change d'avis. C'est la même structure que
l'actualisation hyperbolique. Conséquences :

- il n'existe plus de politique stationnaire « optimale » au sens strict de Bellman. Le DQN apprend
  la politique d'un agent qui ré-optimise à chaque pas (« naïf » au sens de la littérature sur
  l'incohérence temporelle). C'est le comportement standard de toutes les architectures
  multi-horizons (voir §7) ;
- on peut le présenter comme une **propriété psychologiquement plausible** (on promet d'aider plus
  tard, puis on ne le fait pas) plutôt que comme un défaut.

## 3. Implémentation : un DQN à deux têtes

Un seul $\gamma$ ne permet pas de reproduire ce comportement en modifiant uniquement la reward : il faut
**séparer les deux flux dans l'apprentissage**. Changements minimaux dans `agent_policies_gt_ais.py` :

| Élément | Aujourd'hui | Proposition |
|---|---|---|
| Sortie du réseau | `fc3 → action_size` (Q combiné) | `fc3 → 2 × action_size` : $Q^p$ et $Q^e$ (tronc partagé) |
| Replay buffer | $(s, a, r, s', d)$ | $(s, a, p, \tilde e, s', d)$ : les deux termes séparés |
| Reward reçue par `agent.step` | `rewards[i]` (combinée) | `info['personal_reward'][i]`, `info['empathic_reward'][i]` (déjà renvoyés par `env.step`) |
| Action greedy | $\arg\max_a Q$ | $\arg\max_a\,(1-\alpha)Q^p + \alpha Q^e$ |
| Cible TD | $r + \gamma \max_{a'} Q_{\text{tgt}}(s',a')$ | $a^* = \arg\max_{a'}$ du Q **combiné** cible, puis $y^p = p + \gamma_s Q^p_{\text{tgt}}(s',a^*)$ et $y^e = \tilde e + \gamma_o Q^e_{\text{tgt}}(s',a^*)$ |
| Loss | MSE($Q$, $y$) | MSE($Q^p$, $y^p$) + MSE($Q^e$, $y^e$) |

**Point important :** les deux têtes doivent bootstrapper sur **la même action $a^*$**, celle que
l'agent jouerait vraiment, choisie sur le Q combiné. Si chaque tête prenait son propre $\max$, $Q^p$
serait la valeur d'un agent purement égoïste, $Q^e$ celle d'un agent purement altruiste, et leur
somme ne correspondrait à aucune politique réelle (c'est l'écueil connu de l'architecture
*Hybrid Reward*, van Seijen et al., 2017).

Le Q-learning tabulaire se traite de la même façon avec deux tables. Les politiques fixes ne sont
pas concernées. La baseline d'aversion à l'inégalité de Hughes pourrait recevoir le même traitement,
mais ce n'est pas nécessaire.

Esquisse (non branchée sur la simulation) :

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class TwoHeadDQNNetwork(nn.Module):
    # même tronc que DQNNetwork, deux têtes : Q^p (soi) et Q^e (autres)
    def __init__(self, state_size, action_size, hidden_size=64):
        super().__init__()
        self.action_size = action_size
        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 2 * action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        q = self.fc3(x)
        return q[:, :self.action_size], q[:, self.action_size:]   # (Q^p, Q^e)


def two_head_targets(net_target, personal, empathic, next_states, dones,
                     alpha, gamma_self, gamma_others):
    # cibles TD des deux têtes, bootstrappées sur la MÊME action a*
    with torch.no_grad():
        qp_next, qe_next = net_target(next_states)
        q_next = (1 - alpha) * qp_next + alpha * qe_next          # Q combiné
        a_star = q_next.argmax(dim=1, keepdim=True)                 # action réellement jouée
        y_p = personal + gamma_self * qp_next.gather(1, a_star) * (1 - dones)
        y_e = empathic + gamma_others * qe_next.gather(1, a_star) * (1 - dones)
    return y_p, y_e


def two_head_loss(net, net_target, states, actions, personal, empathic,
                  next_states, dones, alpha, gamma_self, gamma_others):
    qp, qe = net(states)
    y_p, y_e = two_head_targets(net_target, personal, empathic, next_states, dones,
                                alpha, gamma_self, gamma_others)
    return F.mse_loss(qp.gather(1, actions), y_p) + F.mse_loss(qe.gather(1, actions), y_e)

**Test d'emboîtement.** Si $\gamma_o = \gamma_s$, la cible combinée $(1-\alpha)y^p + \alpha y^e$ doit être identique
à la cible du DQN actuel calculée sur la reward combinée. On le vérifie sur des tenseurs aléatoires :

In [ ]:
torch.manual_seed(0)
B, S, A, alpha, g = 256, 3, 2, 0.5, 0.99
net_t = TwoHeadDQNNetwork(S, A)
p = torch.rand(B, 1); e = torch.rand(B, 1)
s2 = torch.rand(B, S); d = (torch.rand(B, 1) < 0.05).float()

y_p, y_e = two_head_targets(net_t, p, e, s2, d, alpha, g, g)
combined_two_head = (1 - alpha) * y_p + alpha * y_e

# cible du DQN actuel : r + γ max_a' Q(s', a') avec r et Q combinés
with torch.no_grad():
    qp2, qe2 = net_t(s2)
    q2 = (1 - alpha) * qp2 + alpha * qe2
    single = ((1 - alpha) * p + alpha * e) + g * q2.max(1, keepdim=True)[0] * (1 - d)

print("écart max (γ_o = γ_s) :", float((combined_two_head - single).abs().max()))

y_p, y_e = two_head_targets(net_t, p, e, s2, d, alpha, g, 0.9)
print("écart max (γ_o = 0.9) :", float(((1 - alpha) * y_p + alpha * y_e - single).abs().max()))

## 4. Brute ou normalisée : que faut-il garder fixe ?

Une valeur actualisée a une magnitude d'ordre $\frac{\bar r}{1-\gamma}$. Avec $\gamma_s = 0.99$ et $\gamma_o = 0.9$, $Q^p$ vaut
environ $100\,\bar p$ et $Q^e$ environ $10\,\bar e$. Baisser $\gamma_o$ change donc aussi la part globale de
l'empathie. Deux façons de combiner les têtes :

- **(a) Brute** : $Q = (1-\alpha)Q^p + \alpha Q^e$.
- **(b) Normalisée** : $Q = (1-\alpha)(1-\gamma_s)\,Q^p + \alpha(1-\gamma_o)\,Q^e$. Chaque tête est exprimée en
  « reward moyenne par pas ».

**Ce qui compte pour le comportement, ce sont les différences de Q entre actions, pas le niveau
de Q.** Une conséquence $\delta$ sur $\tilde e$, arrivant $k$ pas après l'action, pèse $\alpha\,\gamma_o^{\,k}\,\delta$ en brut et
$\alpha(1-\gamma_o)\,\gamma_o^{\,k}\,\delta$ en normalisé. Les deux versions **fixent donc des choses différentes** :

| Version | Ce qu'elle garde égal à $\alpha$ | Poids effectif de l'empathie à l'horizon $k$ |
|---|---|---|
| (a) brute | le poids **au présent** : $\alpha_{\text{eff}}(0) = \alpha$ | $\dfrac{\alpha\,\gamma_o^{\,k}}{(1-\alpha)\,\gamma_s^{\,k} + \alpha\,\gamma_o^{\,k}}$ |
| (b) normalisée | le poids **total** (sur un flux de reward constant) | $\dfrac{\alpha(1-\gamma_o)\,\gamma_o^{\,k}}{(1-\alpha)(1-\gamma_s)\,\gamma_s^{\,k} + \alpha(1-\gamma_o)\,\gamma_o^{\,k}}$ |

γ_o garde donc un effet après normalisation : la normalisation conserve l'aire totale, mais
**redistribue cette aire dans le temps**. Les deux versions ne coïncident qu'aux extrêmes :

- $\gamma_o = \gamma_s$ : la normalisation n'est qu'un facteur constant commun, la politique est identique ;
- $\gamma_o = 0$ : l'empathie est inerte dans les deux cas, puisque mon action n'agit pas sur $\tilde e$ au pas
  courant (§5).

In [ ]:
# part de l'empathie dans le poids total, pour un flux de reward constant de 1,
# et α' qu'il faudrait à un agent à γ unique (= γ_s) pour avoir la même part
for g_o in GAMMA_OTHERS:
    raw_s, raw_o = (1 - ALPHA) / (1 - GAMMA_SELF), ALPHA / (1 - g_o)
    share_raw = raw_o / (raw_s + raw_o)
    print(f"γ_o = {g_o:<5}  part de l'empathie : brute = {share_raw:5.1%}   normalisée = {ALPHA:5.1%}"
          f"   →  α' apparié = {share_raw:.3f}")

In [ ]:
def alpha_eff_norm(k, alpha, g_s, g_o):
    # même chose que alpha_eff, avec chaque tête ramenée à une reward moyenne par pas
    w_o = alpha * (1 - g_o) * g_o ** k
    w_s = (1 - alpha) * (1 - g_s) * g_s ** k
    return w_o / (w_s + w_o)

def crossover(g_s, g_o):
    # horizon où la version normalisée repasse sous α : (1-γ_o)γ_o^k = (1-γ_s)γ_s^k
    return np.log((1 - g_s) / (1 - g_o)) / np.log(g_o / g_s)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, fn, title in [(axes[0], alpha_eff, "(a) brute : le présent est fixé à α"),
                      (axes[1], alpha_eff_norm, "(b) normalisée : le total est fixé, le présent est gonflé")]:
    for g_o, c in zip(GAMMA_OTHERS, colors):
        label = rf"$\gamma_o$ = {g_o}" + ("  (actuel)" if g_o == GAMMA_SELF else "")
        ax.plot(k, fn(k, ALPHA, GAMMA_SELF, g_o), color=c, lw=2, label=label)
    ax.axhline(ALPHA, color=INK["muted"], lw=0.8, ls=":")
    ax.set_title(title, loc="left", fontsize=10, color=INK["primary"])
    ax.set_xlabel("horizon k (pas)")
axes[0].set_ylabel(r"$\alpha_{\mathrm{eff}}(k)$")
axes[0].set_ylim(0, 1.0)
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

print("version normalisée :")
for g_o in GAMMA_OTHERS[1:]:
    print(f"  γ_o = {g_o:<5}  α_eff(0) = {alpha_eff_norm(0, ALPHA, GAMMA_SELF, g_o):5.1%}"
          f"   repasse sous α à k ≈ {crossover(GAMMA_SELF, g_o):4.1f} pas")

**Lecture.** La version normalisée ne correspond pas à l'intuition de départ : pour conserver le
poids total, elle rend l'agent **hyper-empathique sur l'immédiat** (≈ 91 % à $k=0$ pour $\gamma_o = 0.9$), puis
égoïste au-delà d'environ 24 pas. La version brute dit exactement « même souci des autres
maintenant, mais il s'éteint plus vite que le souci de soi ». **La baisse du poids total n'y est pas
un artefact : c'est le phénomène lui-même.**

**Décision (02/10/2026) : pas de normalisation.** On garde la version brute, et on contrôle la
confusion avec $\alpha$ par une condition contrôle :

- **Manipulation** : version brute, $\gamma_o < \gamma_s$.
- **Contrôle « α apparié »** : un agent à $\gamma$ unique ($\gamma_o = \gamma_s$) avec un $\alpha'$ choisi pour donner **la même
  part totale d'empathie** (colonne $\alpha'$ ci-dessus, par ex. $\alpha' \approx 0.09$ pour $\gamma_o = 0.9$).

Baisser $\alpha$ réduit l'empathie **uniformément à tous les horizons**. Baisser $\gamma_o$ ne coupe **que le futur**.
Si la manipulation et son contrôle apparié se comportent différemment, c'est l'**horizon** de
l'empathie qui compte, et pas seulement sa quantité. La version normalisée est écartée : elle ne correspond pas
à l'intuition de départ et ne sera pas implémentée.

*Limite du contrôle* : l'appariement se fait sur un flux de reward constant. Les vrais flux $p$ et $\tilde e$
ne sont pas constants, donc la part réelle de l'empathie peut différer un peu. On peut la mesurer
a posteriori sur les runs (rapport des $Q^e$ et $Q^p$ appris).

L'esquisse du §3 est déjà la version brute (poids $1-\alpha$ et $\alpha$ dans `q_next` et l'action greedy).

## 5. Pourquoi c'est un test direct du mécanisme dans notre environnement

Dans l'environnement actuel, **mon action au pas $t$ n'a aucun effet sur l'émotion des autres au pas
$t$** : il n'y a pas de concurrence par pas, l'émotion d'un agent ne dépend que de ses propres
repas. Le seul chemin passe par **la ressource** : je mange, la ressource baisse, la probabilité de
succès des autres baisse, leurs émotions baissent **plus tard** (et l'épisode s'arrête plus tôt à
l'épuisement, ce qui coupe le flux $\tilde e \ge 0$).

Conséquence directe :

- à $\gamma_o \to 0$, $Q^e(s,a)$ ne dépend plus de $a$ et l'empathie **n'a plus aucun effet sur le
  comportement**. En version brute, elle se réduit à diminuer le poids du personnel ($1-\alpha$) ;
- donc **$\gamma_o$ est un curseur qui mesure de combien d'horizon l'empathie a besoin pour produire la
  retenue**. Si les effets de $\alpha$ (soutenabilité, renversement de fairness) disparaissent dès
  $\gamma_o \approx 0.9$ (horizon ≈ 10 pas), on sait qu'ils passent par les conséquences proches. S'ils
  ne disparaissent qu'avec $\gamma_o$ très faible, ils passent par l'effet de fin d'épisode (survie de la
  ressource).

Cela rejoint le point ouvert du tracker (« la retenue ne peut s'apprendre que par ses conséquences
futures ; le lien ma consommation → émotion des autres est très faible »). $\gamma_o$ permet de **quantifier**
ce lien au lieu de seulement le constater.

*Prédiction honnête :* vu la faiblesse du lien (1 unité sur 500), il est probable qu'avec $\gamma_o$ faible
l'empathie devienne comportementalement quasi inerte. C'est un résultat en soi (« une empathie
myope ne protège pas les communs »), mais il faut l'anticiper dans l'interprétation.

## 6. Conditions envisageables

Paramètres ajoutés à `make_condition` : `gamma_others=None` (None → égal à $\gamma$, donc toutes les expériences
existantes sont inchangées).

| Expérience | Conditions | Question |
|---|---|---|
| Horizon de l'empathie (manipulation) | D (voit les émotions, $\alpha=0.5$), $\gamma_s = 0.99$, $\gamma_o \in \{0.99, 0.95, 0.9, 0.8, 0\}$, version brute | Quel horizon empathique faut-il pour obtenir la retenue ? |
| Côté $\gamma_o > \gamma_s$ (à ajouter, voir §8) | même chose avec $\gamma_o \in \{0.995, 0.999\}$ | Une empathie plus patiente que l'intérêt propre (ce que suggèrent les données humaines) renforce-t-elle la retenue ? |
| Contrôle α apparié | D à $\gamma$ unique 0.99, $\alpha' \in \{0.167,\ 0.091,\ 0.048,\ 0.010\}$ (même part totale que $\gamma_o$ = 0.95 / 0.9 / 0.8 / 0) | L'effet de $\gamma_o$ dépasse-t-il un simple « α plus petit » ? |
| Croisement (option) | $\gamma_o$ × see_emotions | L'observation des émotions compense-t-elle une empathie myope ? |

Les $\alpha'$ du contrôle recoupent en partie la grille de l'exp3 ($\alpha \in \{0, 0.25, \dots\}$) : les points bas de
la dose-réponse (α ≈ 0.01–0.17) sont justement ceux qui manquent aujourd'hui.

Cela s'insérerait naturellement dans l'**axe 2 (motivation)** du plan du papier : la motivation
empathique aurait deux dimensions, son **intensité** ($\alpha$) et son **horizon** ($\gamma_o$).

Coût (manipulation + contrôle) : (5 + 4) conditions × 20 seeds = 180 runs, du même ordre que l'exp3.

## 7. Points ouverts et littérature

**À trancher :**
1. ~~Brute ou normalisée (§4) ?~~ → **tranché le 02/10/2026 : brute, sans normalisation**, avec le
   contrôle « α apparié ».
2. Tronc partagé ou deux réseaux séparés ? Le tronc partagé est plus économe et reste comparable au
   DQN actuel (même nombre de couches cachées).
3. Le bug ouvert de `run_step` (action stockée ≠ action jouée) devrait être corrigé **avant**. Avec deux
   têtes bootstrappées sur $a^*$, des transitions mal étiquetées brouilleraient encore plus la tête
   empathique, dont le signal est déjà faible.
4. Variante « hyperbolique » : remplacer $\gamma_o$ par une somme de plusieurs têtes à différents $\gamma$
   (Fedus et al., 2019). C'est plus réaliste psychologiquement, mais aussi plus lourd. À garder pour
   plus tard.

**Littérature (références à vérifier avant de les citer) :**
- *Plusieurs discount factors dans un même agent* : van Seijen et al. 2017, *Hybrid Reward
  Architecture* (décomposition de la reward en têtes) ; Romoff et al. 2019, *Separating value
  functions across time-scales* ; Fedus et al. 2019, *Hyperbolic discounting and learning over
  multiple horizons*.
- *Actualisation sociale* : Jones & Rachlin 2006, *Social discounting* (Psych. Science). Attention,
  il s'agit d'une actualisation selon la **distance sociale**, pas selon le temps.
- **Mise en garde sur l'intuition de départ** (détaillée au §8) : les études d'arbitrage intertemporel *pour autrui*
  (par ex. Albrecht et al. 2011, SCAN ; Ziegler & Tunney 2012, PLoS ONE) trouvent plutôt que les
  choix faits **pour** les autres sont **moins** impulsifs que pour soi. Ce n'est pas exactement notre
  question : il s'agit de choisir à la place de quelqu'un, pas de pondérer son bien-être futur dans
  sa propre utilité. Mais un reviewer pourrait l'objecter. L'argument le plus solide pour
  $\gamma_o < \gamma_s$ serait plutôt celui de l'**empathie affective** : elle répond à une détresse perçue
  *maintenant* et s'éteint quand elle n'est plus perçue. C'est plus défendable que « on se fiche de
  l'avenir des autres », et cela colle mieux à notre terme $\tilde e$, qui est une émotion observée.

## 8. Quelle valeur pour $\gamma_o$ ? Ce que disent les données humaines

**Conclusion d'abord : aucune étude ne donne directement $\gamma_o/\gamma_s$. Les deux études les plus proches
vont dans le sens contraire de l'intuition de départ.** Pour des récompenses monétaires, une
récompense différée **pour autrui** perd *moins* vite de sa valeur qu'une récompense différée pour
soi, et cet écart grandit avec la distance sociale. Autrement dit, $\gamma_o \ge \gamma_s$.

### 8.1 van Rijn et al. (2024), *Dev. Cogn. Neurosci.* 70 — [PMC11570222](https://pmc.ncbi.nlm.nih.gov/articles/PMC11570222/)

- 196 adolescents de 14 à 17 ans. Le choix porte sur 2–8 € **pour soi tout de suite** contre 10 € plus tard,
  pour soi, pour un ami ou pour un inconnu. Délais : 2, 4, 30 et 90 jours.
- Aire sous la courbe (AUC) : soi 0,55 (ET 0,23) ; ami 0,46 (0,19) ; inconnu 0,38 (0,19) ;
  $F(2, 390) = 104{,}69$, $\eta^2_p = 0{,}35$.
- **Piège** : la récompense immédiate va toujours à soi. L'AUC mélange donc le **niveau** (combien
  vaut 10 € pour un inconnu, ≈ notre $\alpha$) et la **pente** avec le délai (≈ notre $\gamma_o$). Une AUC plus basse
  pour autrui ne veut pas dire qu'on actualise plus vite le futur d'autrui.
- La pente, elle, est donnée par l'interaction délai × bénéficiaire ($F(6, 2145) = 10{,}94$,
  $\eta^2_p = 0{,}03$) : *« the decrease of SV with increasing delays was strongest for the self, then for the
  friend, and weakest for the unknown other »*. La valeur pour autrui décroît donc **moins vite** avec le
  délai. Réserve : la valeur pour autrui part plus bas (plancher à 0,1), donc une décroissance absolue
  plus faible peut être en partie un effet plancher. Les valeurs délai par délai ne sont données que
  dans une figure (Fig. 3A), pas en chiffres.

### 8.2 Osiński & Karbowski (2017), *PLoS ONE* 12(2) — [PMC5308811](https://pmc.ncbi.nlm.nih.gov/articles/PMC5308811/)

- 161 étudiants. Le choix porte sur une petite somme pour soi contre une somme plus grande pour une
  personne de rang social 1 à 100, avec **le même délai pour les deux** : immédiat, 1 jour, 1 mois ou
  5 ans.
- Table 1 : valeur subjective de la récompense d'autrui relative à la sienne.
- **Ce design permet un calcul propre.** Avec une actualisation exponentielle, la valeur relative au délai
  $D$ vaut $w \cdot e^{-(r_o - r_s)D}$. Le rapport $R = SV(5\text{ ans}) / SV(\text{immédiat})$ élimine le niveau $w$ et donne
  directement l'écart de taux $r_s - r_o = \ln R / 5$ par an.

In [ ]:
# Osiński & Karbowski 2017, PLoS ONE, Table 1 : valeur subjective relative (autrui / soi)
DIST = [1, 5, 20, 60, 100]                       # rang de distance sociale
SV_IMMEDIATE = [0.740, 0.498, 0.253, 0.143, 0.100]
SV_5_YEARS   = [0.742, 0.529, 0.319, 0.208, 0.162]

print("distance   w (≈ α)   R = SV(5 ans)/SV(0)   r_s − r_o (par an)")
for d, s0, s5 in zip(DIST, SV_IMMEDIATE, SV_5_YEARS):
    R = s5 / s0
    print(f"{d:>8}   {s0:7.3f}   {R:19.2f}   {np.log(R) / 5:+18.3f}")

**Lecture.** Pour toutes les distances, $R \ge 1$ : quand on ajoute le même délai aux deux récompenses,
celle d'autrui garde plus de valeur que la sienne. Pour un inconnu (rang 100), l'écart de taux est
d'environ **+0,10 par an en faveur d'autrui**. Pour un proche (rang 1), il est nul.

**Passer à $\gamma$.** Comme on l'a vu, seul le rapport des taux est transposable à nos pas de temps :
$\frac{1-\gamma_o}{1-\gamma_s} \approx \frac{r_o}{r_s} = 1 - \frac{r_s - r_o}{r_s}$. Il faut donc aussi le taux propre $r_s$, que l'étude ne mesure
pas. La cellule suivante montre la sensibilité à ce choix.

In [ ]:
# sensibilité : γ_o implicite pour γ_s = 0.99, selon le taux propre r_s supposé
DELTA_R = {d: np.log(s5 / s0) / 5 for d, s0, s5 in zip(DIST, SV_IMMEDIATE, SV_5_YEARS)}
for r_s in [0.2, 0.5, 1.0]:
    row = []
    for d in [20, 100]:
        ratio = max(1 - DELTA_R[d] / r_s, 0.0)            # r_o / r_s
        row.append(f"rang {d:>3} : γ_o ≈ {1 - (1 - GAMMA_SELF) * ratio:.4f}")
    print(f"r_s = {r_s:3.1f}/an   " + "   ".join(row))

**Ce qu'on peut en tirer pour 03_GT :**

1. **Le sens de l'effet.** Les données humaines sur l'argent soutiennent $\gamma_o \ge \gamma_s$ (de 0,990 à environ 0,995
   selon $r_s$, pour un inconnu), et non $\gamma_o < \gamma_s$. L'explication avancée est la *distance psychologique*
   (construal level) : un délai et une distance sociale rendent tous deux la récompense abstraite, et
   l'égoïsme est surtout une affaire d'immédiat. Les études qui trouvent l'inverse (Takahashi 2007 ;
   van Rijn 2024 si l'on regarde l'AUC seule) confondent souvent le niveau et la pente.
2. **Ce qui reste défendable pour $\gamma_o < \gamma_s$.** Ces études portent sur **de l'argent attribué par un choix**,
   pas sur une **émotion perçue**. Notre $\tilde e$ est une empathie *affective*, une réaction à une détresse
   observée. L'argument « elle s'éteint quand la détresse n'est plus perçue » n'est pas réfuté par ces
   données, mais il n'est pas non plus soutenu par elles. Je n'ai pas trouvé d'étude qui mesure la
   dévaluation temporelle de l'*émotion* d'autrui.
3. **Recommandation.** Ne pas fixer $\gamma_o$ à partir de la littérature. En faire un **paramètre exploré des
   deux côtés de $\gamma_s$**, avec $\gamma_o > \gamma_s$ ajouté à la grille du §6. Citer Osiński & Karbowski (2017) comme
   référence pour le sens « humain », et présenter $\gamma_o < \gamma_s$ comme une hypothèse sur l'empathie
   affective.
4. **Bonus pour $\alpha$.** La même table donne le **niveau** $w$ : environ 0,25 pour une personne de rang 20 et
   environ 0,10 pour un inconnu. Si nos 6 agents sont vus comme des membres d'un même groupe, plutôt que
   des proches, $\alpha$ autour de 0,1–0,25 serait la plage « humainement réaliste ». C'est justement la zone basse
   de la dose-réponse que l'exp3 ne couvre pas bien.

**À compléter :** Osiński et al. (2026, *J. Behav. Decis. Making*) manipulent les délais pour soi et pour
autrui **indépendamment**, ce qui est exactement le design qui donnerait $r_o$ et $r_s$ séparément. Je n'ai eu
accès qu'à l'abstract (article payant) : à récupérer.